# Scripts for measuring noise on a large skypatch image

In [ ]:
# | default_exp euclid.skypatch_noise

In [ ]:
# | exporti

import glob
from pathlib import Path


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt

from scipy.stats import median_abs_deviation

from astropy.io import fits
from astropy.nddata import CCDData
from astropy.stats import sigma_clip
from astropy.visualization import simple_norm


from photutils.aperture import CircularAnnulus

In [ ]:
# | exporti

plt.style.use("nicl.euclid.v1nicl")

In [ ]:
# | export


def measure_noise_in_circular_annuli(
    image_path,
    object_mask_path,
    profile_path,
    hdul_index=0,
    pixelscale=0.3,
    core_mask_path=None,
    rad_limit_annulus=None,
    verbose=None,
    num_points=1000,
    bkg_region_in_arcsec=300,
    output_path=None,
    show_diagnostic_plots=True,
    save_diagnostics=False,
    show_annuli=False,
    zp=23.9,
):
    profiles = sorted(glob.glob(str(profile_path)))
    print(f"Profile: {profiles}")

    df_dict = {}
    for profile in profiles:
        filekey = profile.split("/")[-1].split("_")
        clusterid = filekey[0] + "_" + filekey[1]
        boxsize = (filekey[5].split(".")[0]) if len(filekey) > 4 else 0
        gs = f"_{filekey[filekey.index('gscale') + 1]}" if "gscale" in filekey else ""
        label = f"{clusterid}_{boxsize}{gs}"
        df_dict[label] = pd.read_csv(profile, skiprows=1)
        print(f"label is: {label}...")

    image_ccd = CCDData.read(image_path, unit="adu", hdu=hdul_index)
    image = image_ccd.data
    object_mask = fits.getdata(object_mask_path).astype(bool)
    masked_image = np.where(object_mask, np.nan, image)

    print("Masked image is created...")

    prof = df_dict[label]
    prof["R"] = prof["R"] / pixelscale

    if rad_limit_annulus:
        prof = prof[prof["R"] < rad_limit_annulus]

    selected_annuli = sorted(set(prof["R"].values))

    first_radius = selected_annuli[0] if selected_annuli else 0
    r_max = max(selected_annuli) if selected_annuli else 0

    image_height, image_width = image.shape
    valid_y, valid_x = np.where(~object_mask)
    valid_points = np.column_stack((valid_x, valid_y))
    valid_points = valid_points[
        (valid_points[:, 0] > r_max)
        & (valid_points[:, 0] < image_width - r_max)
        & (valid_points[:, 1] > r_max)
        & (valid_points[:, 1] < image_height - r_max)
    ]

    flux_stats = []
    final_selected_points = []

    while len(final_selected_points) < num_points and valid_points.size > 0:
        x, y = valid_points[np.random.choice(len(valid_points))]
        x += np.random.uniform(-0.5, 0.5)
        y += np.random.uniform(-0.5, 0.5)

        annulus = CircularAnnulus((x, y), r_in=0.5, r_out=first_radius)
        test = annulus.to_mask(method="exact").multiply(masked_image)
        if np.count_nonzero(np.isfinite(test) & (test != 0)) == 0:
            continue

        final_selected_points.append((x, y))
        print(f"The noise profile is being extracted around point: {x:.1f}, {y:.1f}")

        for i in range(len(selected_annuli) - 1):
            r_in, r_out = selected_annuli[i], selected_annuli[i + 1]
            ann = (
                CircularAnnulus((x, y), r_in, r_out)
                .to_mask(method="center")
                .to_image(masked_image.shape)
            )
            values = masked_image[np.isfinite(masked_image) & (ann > 0)]

            stats = {
                "Centre_pixel": (x, y),
                "Inner_Radius_pix": r_in,
                "Outer_Radius_pix": r_out,
                "Inner_Radius_arcsec": r_in * pixelscale,
                "Outer_Radius_arcsec": r_out * pixelscale,
                "SMA_annulus_centre_pix": (r_in + r_out) / 2,
                "SMA_annulus_centre_arcsec": ((r_in + r_out) * pixelscale) / 2,
            }

            if len(values) == 0:
                stats.update(
                    {
                        key: np.nan
                        for key in [
                            "Mean_flux_annulus",
                            "Median_flux_annulus",
                            "Std_flux_annulus",
                            "Clipped_mean_flux_annulus",
                            "Clipped_median_flux_annulus",
                            "Clipped_Std_flux_annulus",
                            "Total_valid_pix_annulus",
                        ]
                    }
                )
            else:
                clipped = sigma_clip(values, sigma=3, cenfunc="median", maxiters=5)
                clipped_values = clipped.data[~clipped.mask]
                stats.update(
                    {
                        "Mean_flux_annulus": np.nanmean(values) / pixelscale**2,
                        "Median_flux_annulus": np.nanmedian(values) / pixelscale**2,
                        "Std_flux_annulus": np.nanstd(values)
                        if len(values) > 1
                        else np.nan,
                        "Clipped_mean_flux_annulus": np.nanmean(clipped_values)
                        / pixelscale**2,
                        "Clipped_median_flux_annulus": np.nanmedian(clipped_values)
                        / pixelscale**2,
                        "Clipped_Std_flux_annulus": np.nanstd(clipped_values)
                        if len(clipped_values) > 1
                        else np.nan,
                        "Total_valid_pix_annulus": len(values),
                    }
                )

            flux_stats.append(stats)

    flux_stats_df = pd.DataFrame(flux_stats)

    # Background subtraction
    flux_grouped = flux_stats_df.groupby("Centre_pixel")

    flux_combined = []
    for centre, group in flux_grouped:
        bkg_region = group[group["SMA_annulus_centre_arcsec"] > bkg_region_in_arcsec]
        if bkg_region.empty:
            continue
        bkg = np.nanmean(bkg_region["Clipped_median_flux_annulus"])
        noise = np.nanstd(bkg_region["Clipped_median_flux_annulus"])
        group = group.copy()
        group["bkg_sub_flux"] = group["Clipped_median_flux_annulus"] - bkg
        group["bkg_noise"] = 3 * noise
        flux_combined.append(group)

    extended_measurement_table = pd.concat(flux_combined, ignore_index=True)

    # Median Absolute Deviation calculation as noise
    mad_group = extended_measurement_table.groupby("SMA_annulus_centre_arcsec")
    radii = sorted(mad_group.groups.keys())
    noise_measurements = pd.DataFrame(
        {
            "SMA_annulus_centre_arcsec": [r for r in radii],
            "MAD_Median_Clipped_Flux": [
                median_abs_deviation(
                    mad_group.get_group(r)["Clipped_median_flux_annulus"],
                    scale="normal",
                    nan_policy="omit",
                )
                for r in radii
            ],
            "MAD_Bkg_Subtracted_Flux": [
                median_abs_deviation(
                    mad_group.get_group(r)["bkg_sub_flux"],
                    scale="normal",
                    nan_policy="omit",
                )
                for r in radii
            ],
        }
    )

    if output_path:
        output_path = Path(output_path)
        output_path.mkdir(parents=True, exist_ok=True)
        extended_measurement_table.to_csv(
            output_path / f"{verbose}_noise_stats_extended_datatable.csv", index=False
        )
        noise_measurements.to_csv(
            output_path / f"{verbose}_noise_measurements.csv", index=False
        )

    if show_annuli and final_selected_points:
        print(
            "Plotting the annuli overlays on the image... May take a while if your image is big, and/or if you have lots of annuli... :) "
        )
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(
            masked_image,
            origin="lower",
            cmap="gray",
            norm=simple_norm(masked_image, "sqrt", percent=99),
        )
        ax.set_title("Overlay of All Fitted Annuli")

        for x, y in final_selected_points:
            for i in range(len(selected_annuli) - 1):
                r_out = selected_annuli[i + 1]
                ring = plt.Circle(
                    (x, y),
                    radius=r_out,
                    edgecolor="magenta",
                    facecolor="none",
                    linewidth=0.4,
                    alpha=0.5,
                )
                ax.add_patch(ring)

        ax.set_xlim(0, masked_image.shape[1])
        ax.set_ylim(0, masked_image.shape[0])
        plt.show()

    if show_diagnostic_plots:
        noise_profile_diagnostics(
            extended_measurement_table,
            verbose=verbose,
            save_plots=save_diagnostics,
            output_path=output_path,
            zp=zp,
        )

    return extended_measurement_table, noise_measurements, label

In [ ]:
# | export


def noise_profile_diagnostics(
    extended_measurement_table,
    verbose=None,
    output_path=None,
    save_plots=True,
    zp=23.9,
):
    grouped = extended_measurement_table.groupby("Centre_pixel")

    def plot_profiles(ax, x, y, ylabel):
        ax.plot(x, y, marker="o", markersize=2)
        ax.hlines(0, xmin=np.min(x), xmax=np.max(x), color="k", lw=1, ls="-")
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_xlabel("Radius (arcsec)", fontsize=14)
        ax.set_ylim(-0.2, 0.2)

    # Plot: Clipped median & mean flux profiles
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    for _, group in grouped:
        plot_profiles(
            ax[0],
            group["SMA_annulus_centre_arcsec"],
            group["Clipped_median_flux_annulus"],
            "Clipped Median Flux",
        )
        plot_profiles(
            ax[1],
            group["SMA_annulus_centre_arcsec"],
            group["Clipped_mean_flux_annulus"],
            "Clipped Mean Flux",
        )
    fig.tight_layout()
    if save_plots:
        plt.savefig(Path(output_path) / f"{verbose}_Median_Mean_Flux_CLIPPED.pdf")
    plt.show()

    # Plot: Background-subtracted flux profiles
    fig, ax = plt.subplots(figsize=(4, 4))
    for _, group in grouped:
        ax.plot(
            group["SMA_annulus_centre_arcsec"],
            group["bkg_sub_flux"],
            marker="o",
            markersize=2,
        )
        ax.hlines(
            group["bkg_noise"].iloc[0],
            xmin=np.min(group["SMA_annulus_centre_arcsec"]),
            xmax=np.max(group["SMA_annulus_centre_arcsec"]),
            color="k",
            lw=0.2,
            ls="--",
        )
    ax.hlines(
        0,
        xmin=0,
        xmax=np.max(extended_measurement_table["SMA_annulus_centre_arcsec"]),
        color="k",
        lw=0.5,
        ls="-",
    )
    ax.set_ylabel("Background-subtracted Median Flux", fontsize=12)
    ax.set_xlabel("Radius (arcsec)", fontsize=12)
    fig.tight_layout()
    if save_plots:
        plt.savefig(Path(output_path) / f"{verbose}_BkgsubFlux.pdf")
    plt.show()

    # Plot: MAD profiles in flux and magnitude space
    flux_grouped_rad = extended_measurement_table.groupby("SMA_annulus_centre_arcsec")
    radii = sorted(flux_grouped_rad.groups.keys())

    mad_median_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["Median_flux_annulus"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]
    mad_clipped_median_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["Clipped_median_flux_annulus"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]
    mad_bkg_sub_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["bkg_sub_flux"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]

    fig, ax = plt.subplots(1, 2, figsize=(8, 4))

    # Plot in flux units
    ax[0].plot(radii, np.array(mad_median_flux) * 3, label="MAD of Median Flux")
    ax[0].plot(
        radii, np.array(mad_clipped_median_flux) * 3, label="MAD of Clipped Median Flux"
    )
    ax[0].plot(
        radii, np.array(mad_bkg_sub_flux) * 3, label="MAD of Bkg Subtracted Flux"
    )
    ax[0].set_ylabel("3×MAD (Flux / arcsec²)", fontsize=14)
    ax[0].set_xscale("log")
    ax[0].grid(True)
    ax[0].legend()
    ax[0].set_ylim(-0.2, 2)

    # Plot in magnitude units
    def flux_to_mag(flux):
        return zp - 2.5 * np.log10(flux)

    ax[1].plot(
        radii, flux_to_mag(np.array(mad_median_flux) * 3), label="MAD of Median Flux"
    )
    ax[1].plot(
        radii,
        flux_to_mag(np.array(mad_clipped_median_flux) * 3),
        label="MAD of Clipped Median Flux",
    )
    ax[1].plot(
        radii,
        flux_to_mag(np.array(mad_bkg_sub_flux) * 3),
        label="MAD of Bkg Subtracted Flux",
    )
    ax[1].invert_yaxis()
    ax[1].set_ylabel(r"3×MAD [mag / $\rm arcsec^2$]", fontsize=14)
    ax[1].set_xscale("log")
    ax[1].grid(True)
    ax[1].legend()

    for a in ax:
        a.set_xlabel("Radius (arcsec)", fontsize=14)

    fig.tight_layout()
    if save_plots:
        plt.savefig(Path(output_path) / f"{verbose}_MAD_profiles.pdf")
    plt.show()

    print("Finished plotting noise diagnostics.")